# Proyecto completo en una sola libreta

Esta libreta hace todo el flujo del proyecto:
1. Descarga y prepara el dataset binario Healthy vs Rotten.
2. Entrena un clasificador YOLO11-cls.
3. Evalua en val y test.
4. Ejecuta una prueba automatica con una imagen.
5. Visualiza metricas con matplotlib.
6. Ejecuta inferencia sobre una imagen o una carpeta.

In [ ]:
%pip install -q kagglehub tqdm ultralytics opencv-python matplotlib pandas scikit-learn

In [ ]:
from pathlib import Path
import os
import random
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO
import kagglehub

SEED = 42
random.seed(SEED)

DATASET_SLUG = "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten"
DATA_DIR = Path.cwd() / "fruit_binary_yolo_cls"
BASE_MODEL = "yolo11n-cls.pt"
TRAIN_RUN_NAME = "fruit_hs_vs_rt_cls_simple_ft5"
DETECTOR_WEIGHTS = "yolo11n.pt"
CLASSIFIER_WEIGHTS = Path("runs/classify") / TRAIN_RUN_NAME / "weights" / "best.pt"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PRODUCE_CLASS_NAMES = {"apple", "banana", "orange", "broccoli", "carrot"}

if torch.cuda.is_available():
    DEVICE = 0
    print("Usando GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("Usando CPU")

print("Data dir:", DATA_DIR)
print("Base model:", BASE_MODEL)
print("Run name:", TRAIN_RUN_NAME)

In [ ]:
def find_dataset_root(download_path):
    candidates = [download_path] + [path for path in download_path.iterdir() if path.is_dir()]
    for candidate in candidates:
        subdirs = [path.name for path in candidate.iterdir() if path.is_dir()]
        has_healthy = any(name.endswith("_Healthy") for name in subdirs)
        has_rotten = any(name.endswith("_Rotten") for name in subdirs)
        if has_healthy and has_rotten:
            return candidate
    raise FileNotFoundError("No se encontro una carpeta con estructura *_Healthy / *_Rotten")

def collect_images(dataset_root):
    grouped_paths = defaultdict(list)
    for class_dir in sorted(path for path in dataset_root.iterdir() if path.is_dir()):
        name = class_dir.name
        if name.endswith("_Healthy"):
            label = "Healthy"
        elif name.endswith("_Rotten"):
            label = "Rotten"
        else:
            continue

        files = [path for path in class_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTS]
        grouped_paths[label].extend(files)
    return grouped_paths

def stratified_split(items_by_label, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9
    rng = random.Random(seed)

    split = {
        "train": defaultdict(list),
        "val": defaultdict(list),
        "test": defaultdict(list),
    }

    for label, items in items_by_label.items():
        items = items.copy()
        rng.shuffle(items)

        count = len(items)
        count_train = int(count * train_ratio)
        count_val = int(count * val_ratio)

        split["train"][label] = items[:count_train]
        split["val"][label] = items[count_train:count_train + count_val]
        split["test"][label] = items[count_train + count_val:]

    return split

def link_or_copy(src, dst):
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def build_binary_dataset(output_root=DATA_DIR):
    raw_download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    dataset_root = find_dataset_root(raw_download_path)
    grouped_paths = collect_images(dataset_root)
    splits = stratified_split(grouped_paths, seed=SEED)

    if output_root.exists():
        shutil.rmtree(output_root)

    for split_name in ["train", "val", "test"]:
        for label in ["Healthy", "Rotten"]:
            (output_root / split_name / label).mkdir(parents=True, exist_ok=True)

    jobs = []
    for split_name, split_labels in splits.items():
        for label, paths in split_labels.items():
            out_dir = output_root / split_name / label
            for index, src in enumerate(paths):
                dst = out_dir / f"{label.lower()}_{index:07d}{src.suffix.lower()}"
                jobs.append((src, dst))

    with ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 8)) as executor:
        list(tqdm(executor.map(lambda item: link_or_copy(*item), jobs), total=len(jobs), desc="Preparando dataset"))

    summary = {}
    for split_name in ["train", "val", "test"]:
        summary[split_name] = {}
        for label in ["Healthy", "Rotten"]:
            count = len(list((output_root / split_name / label).glob("*")))
            summary[split_name][label] = count

    return summary

def count_images(root):
    summary = {}
    for split_name in ["train", "val", "test"]:
        summary[split_name] = {}
        for label in ["Healthy", "Rotten"]:
            label_dir = root / split_name / label
            count = len([path for path in label_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTS])
            summary[split_name][label] = count
    return summary

def choose_sample_image(test_root=DATA_DIR / "test"):
    for label in ["Healthy", "Rotten"]:
        label_dir = test_root / label
        if not label_dir.exists():
            continue
        for path in sorted(label_dir.glob("*")):
            if path.suffix.lower() in IMAGE_EXTS:
                return path
    raise FileNotFoundError("No se encontro una imagen de prueba en DATA_DIR/test")

def metrics_to_dataframe(split_name, metrics):
    rows = []
    results_dict = getattr(metrics, "results_dict", {}) or {}
    for key, value in results_dict.items():
        if isinstance(value, (int, float)):
            rows.append({
                "split": split_name,
                "metric": key,
                "value": float(value),
            })
    return pd.DataFrame(rows)

In [ ]:
dataset_summary = build_binary_dataset()
dataset_summary

In [ ]:
model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_DIR),
    task="classify",
    imgsz=640,
    epochs=5,
    batch=32,
    device=DEVICE,
    project="runs/classify",
    name=TRAIN_RUN_NAME,
    pretrained=True,
    seed=SEED,
    deterministic=True,
    verbose=True,
    workers=0,
    exist_ok=True,
)

best_weights = Path("runs/classify") / TRAIN_RUN_NAME / "weights" / "best.pt"
assert best_weights.exists(), f"No existe best.pt en {best_weights}"
print("Best weights:", best_weights)

In [ ]:
if "best_weights" not in globals():
    best_weights = Path("runs/classify") / TRAIN_RUN_NAME / "weights" / "best.pt"

classifier_model = YOLO(str(best_weights))
val_metrics = classifier_model.val(data=str(DATA_DIR), split="val", imgsz=640, device=DEVICE)
test_metrics = classifier_model.val(data=str(DATA_DIR), split="test", imgsz=640, device=DEVICE)

metrics_df = pd.concat([
    metrics_to_dataframe("val", val_metrics),
    metrics_to_dataframe("test", test_metrics),
], ignore_index=True)

metrics_df

In [ ]:
if metrics_df.empty:
    print("No se encontraron metricas numericas para graficar.")
else:
    pivot_df = metrics_df.pivot(index="metric", columns="split", values="value")
    ax = pivot_df.plot(kind="bar", figsize=(12, 6))
    ax.set_title("Metricas de validacion y test")
    ax.set_ylabel("Valor")
    ax.set_xlabel("Metrica")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
detector = YOLO(DETECTOR_WEIGHTS)
classifier = YOLO(str(best_weights))

def clamp_box(x1, y1, x2, y2, width, height):
    x1 = max(0, min(int(x1), width - 1))
    y1 = max(0, min(int(y1), height - 1))
    x2 = max(0, min(int(x2), width - 1))
    y2 = max(0, min(int(y2), height - 1))
    if x2 <= x1:
        x2 = min(width - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(height - 1, y1 + 1)
    return x1, y1, x2, y2

def detect_and_count(image_path, det_conf=0.20, det_iou=0.60, min_box_area=32 * 32, only_produce=True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(f"No se pudo leer imagen: {image_path}")

    height, width = bgr.shape[:2]
    det_result = detector.predict(
        source=bgr,
        conf=det_conf,
        iou=det_iou,
        device=DEVICE,
        verbose=False,
    )[0]

    instances = []
    healthy_count = 0
    rotten_count = 0
    names = det_result.names

    for box in det_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1, y1, x2, y2 = clamp_box(x1, y1, x2, y2, width, height)

        area = (x2 - x1) * (y2 - y1)
        if area < min_box_area:
            continue

        class_index = int(box.cls[0].item())
        det_name = names[class_index]
        det_score = float(box.conf[0].item())

        if only_produce and PRODUCE_CLASS_NAMES is not None and det_name not in PRODUCE_CLASS_NAMES:
            continue

        crop = bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        cls_result = classifier.predict(source=crop, imgsz=640, device=DEVICE, verbose=False)[0]
        top_index = int(cls_result.probs.top1)
        health_label = cls_result.names[top_index]
        health_conf = float(cls_result.probs.top1conf.item())

        label_lower = health_label.lower()
        if "healthy" in label_lower:
            healthy_count += 1
            box_color = (60, 180, 75)
        elif "rotten" in label_lower:
            rotten_count += 1
            box_color = (230, 25, 75)
        else:
            box_color = (255, 165, 0)

        text = f"{health_label} {health_conf:.2f} | {det_name} {det_score:.2f}"
        cv2.rectangle(bgr, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(bgr, text, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2, cv2.LINE_AA)

        instances.append({
            "bbox_xyxy": (x1, y1, x2, y2),
            "det_class": det_name,
            "det_conf": det_score,
            "health_label": health_label,
            "health_conf": health_conf,
        })

    summary = {
        "healthy": healthy_count,
        "rotten": rotten_count,
        "total": healthy_count + rotten_count,
    }

    return bgr, instances, summary

In [ ]:
IMAGE_PATH = choose_sample_image()
OUTPUT_IMAGE_PATH = Path("sample_result.jpg")

print("Imagen de prueba:", IMAGE_PATH)

annotated_bgr, instances, summary = detect_and_count(
    IMAGE_PATH,
    det_conf=0.20,
    det_iou=0.60,
    min_box_area=32 * 32,
    only_produce=True,
)

cv2.imwrite(str(OUTPUT_IMAGE_PATH), annotated_bgr)

print(f"Conteo final -> {summary['rotten']} podridas, {summary['healthy']} sanas")
print(f"Instancias analizadas: {summary['total']}")
print("Imagen guardada en:", OUTPUT_IMAGE_PATH)

rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(rgb)
plt.axis("off")
plt.title(f"Resultado: {summary['rotten']} Rotten, {summary['healthy']} Healthy")
plt.show()

pd.DataFrame(instances)

In [ ]:
INPUT_DIR = Path("inference_images")
OUTPUT_DIR = Path("inference_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for image_path in sorted(INPUT_DIR.glob("*")):
    if image_path.suffix.lower() not in IMAGE_EXTS:
        continue

    annotated, instances, summary = detect_and_count(image_path, only_produce=True)
    out_image = OUTPUT_DIR / image_path.name
    cv2.imwrite(str(out_image), annotated)

    rows.append({
        "image": image_path.name,
        "healthy": summary["healthy"],
        "rotten": summary["rotten"],
        "total": summary["total"],
    })

df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / "counts.csv"
df.to_csv(csv_path, index=False)

print("CSV generado en:", csv_path)
df.head()